# 面试题：海量文本近重复检测怎样用 MinHash 与 LSH 实现？

精确两两 Jaccard 是 `O(N²)`，无法处理大规模语料。本 Notebook 手写规范化、word shingles、稳定哈希、MinHash、banding LSH、候选精排、并查集聚类、增量更新、删除和 pair-level 评估。

目标是减少候选而不漏掉近重复；MinHash 只是概率近似，最终合并前仍应做精确相似度与来源/时间规则。

In [ ]:
import copy,hashlib,json,math,re,unicodedata,warnings  # 导入本单元所需的依赖。
from collections import defaultdict  # 导入本单元所需的依赖。
from dataclasses import dataclass  # 导入本单元所需的依赖。
from itertools import combinations  # 导入本单元所需的依赖。
from types import MappingProxyType  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore",message="The pynvml package is deprecated")  # 计算并保存当前步骤的中间状态。
import numpy as np  # 导入本单元所需的依赖。
RNG71=np.random.default_rng(7101)  # 计算并保存当前步骤的中间状态。
PRIME71=4294967311  # 计算并保存当前步骤的中间状态。
def canonical71(x): return json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(",",":"))  # 定义本节可复用的核心函数。
def sha71(x): return hashlib.sha256(x).hexdigest()  # 定义本节可复用的核心函数。
assert PRIME71>2**32  # 用受控断言验证关键不变量。

## 1. 文档规范化、shingle 与版本合同

先 NFKC、小写、折叠空白和词元化，再生成连续 3-token shingles。过短文档回退为整段 token tuple，空文档拒绝。规范化强度决定“重复”的业务语义：去掉数字可能误合并不同版本，保留模板噪声又会漏召回。

每个文档带稳定 ID、版本和 source；同 source 的模板复制与跨 source 抄袭可以采用不同阈值。

In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Doc71: doc_id:str; version:int; source:str; text:str  # 定义承载本节状态与行为的数据结构。
docs71=[  # 计算并保存当前步骤的中间状态。
    Doc71("a",1,"s1","vector search uses an inverted file and product quantization"),  # 执行当前语句以推进本节示例。
    Doc71("b",1,"s2","vector search uses an inverted file and product quantization method"),  # 执行当前语句以推进本节示例。
    Doc71("c",1,"s3","product quantization compresses vectors for fast vector search"),  # 执行当前语句以推进本节示例。
    Doc71("d",1,"s1","graph neural networks aggregate messages from neighboring nodes"),  # 执行当前语句以推进本节示例。
    Doc71("e",1,"s2","graph neural networks aggregate messages from neighbour nodes"),  # 执行当前语句以推进本节示例。
    Doc71("f",1,"s4","a bloom filter is a probabilistic membership structure"),  # 执行当前语句以推进本节示例。
    Doc71("g",1,"s5","a bloom filter is a probabilistic membership data structure"),  # 执行当前语句以推进本节示例。
    Doc71("h",1,"s6","time series forecasting requires chronological validation"),  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。
def tokens71(text): return re.findall(r"[a-z0-9]+|[\u4e00-\u9fff]",unicodedata.normalize("NFKC",text).lower())  # 定义本节可复用的核心函数。
def shingles71(text,n=3):  # 定义本节可复用的核心函数。
    toks=tokens71(text)  # 计算并保存当前步骤的中间状态。
    if not toks: raise ValueError("empty_document")  # 按当前条件选择后续控制路径。
    return {tuple(toks[i:i+n]) for i in range(len(toks)-n+1)} if len(toks)>=n else {tuple(toks)}  # 返回当前分支计算出的结果。
assert len({d.doc_id for d in docs71})==8 and all(d.version==1 for d in docs71)  # 用受控断言验证关键不变量。
assert ("vector","search","uses") in shingles71(docs71[0].text) and len(shingles71("one two"))==1  # 用受控断言验证关键不变量。
try: shingles71(" "); raise AssertionError("empty document accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="empty_document"  # 捕获预期异常并验证失败分支。

## 2. 精确 Jaccard 是最终 oracle

`J(A,B)=|A∩B|/|A∪B|`。MinHash 的碰撞概率等于 Jaccard，但有限签名会有方差，因此 exact Jaccard 用于小规模 gold、候选精排和算法单元测试。

对模板文档还可使用 containment；这里坚持 Jaccard，避免把不同指标混入一个阈值。

In [ ]:
def jaccard71(a,b):  # 定义本节可复用的核心函数。
    a=set(a); b=set(b)  # 计算并保存当前步骤的中间状态。
    if not a and not b: raise ValueError("empty_sets_undefined")  # 按当前条件选择后续控制路径。
    return len(a&b)/len(a|b)  # 返回当前分支计算出的结果。
shingle_map71={d.doc_id:shingles71(d.text) for d in docs71}  # 计算并保存当前步骤的中间状态。
assert jaccard71(shingle_map71["a"],shingle_map71["a"])==1.  # 用受控断言验证关键不变量。
assert jaccard71(shingle_map71["a"],shingle_map71["b"])>jaccard71(shingle_map71["a"],shingle_map71["d"])  # 用受控断言验证关键不变量。
assert 0<=jaccard71(shingle_map71["f"],shingle_map71["g"])<=1  # 用受控断言验证关键不变量。
try: jaccard71(set(),set()); raise AssertionError("empty Jaccard accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="empty_sets_undefined"  # 捕获预期异常并验证失败分支。

## 3. 稳定 shingle hash 与 MinHash

Python 内置 `hash` 跨进程随机，不能用于持久化签名。先用 SHA-256 截取 32 bit，再用 `h_i(x)=(a_i x+b_i) mod p` 构造 64 个置换近似；每维取集合最小值。

参数 `a/b`、prime、shingle recipe 和签名长度必须随制品保存。同 seed 可复现，不同 seed 的签名不能混用。

In [ ]:
def stable_hash71(shingle): return int.from_bytes(hashlib.sha256("\x1f".join(shingle).encode()).digest()[:4],"little")  # 定义本节可复用的核心函数。
NUM_PERM71=64  # 计算并保存当前步骤的中间状态。
A71=RNG71.integers(1,PRIME71,size=NUM_PERM71,dtype=np.uint64); B71=RNG71.integers(0,PRIME71,size=NUM_PERM71,dtype=np.uint64)  # 计算并保存当前步骤的中间状态。
assert A71.shape==(NUM_PERM71,) and B71.shape==(NUM_PERM71,) and bool((A71>0).all())  # 用受控断言验证关键不变量。
def signature71(shingles):  # 定义本节可复用的核心函数。
    values=np.array([stable_hash71(s) for s in shingles],dtype=np.uint64)  # 计算并保存当前步骤的中间状态。
    if values.size==0: raise ValueError("empty_signature")  # 按当前条件选择后续控制路径。
    return ((A71[:,None]*values[None,:]+B71[:,None])%PRIME71).min(1).astype(np.uint64)  # 返回当前分支计算出的结果。
signatures71={i:signature71(s) for i,s in shingle_map71.items()}  # 计算并保存当前步骤的中间状态。
assert signatures71["a"].shape==(64,) and signatures71["a"].dtype==np.uint64  # 用受控断言验证关键不变量。
assert np.array_equal(signatures71["a"],signature71(shingle_map71["a"]))  # 用受控断言验证关键不变量。
estimate_ab71=float(np.mean(signatures71["a"]==signatures71["b"])); exact_ab71=jaccard71(shingle_map71["a"],shingle_map71["b"])  # 计算并保存当前步骤的中间状态。
assert abs(estimate_ab71-exact_ab71)<.25 and stable_hash71(("a","b"))==stable_hash71(("a","b"))  # 用受控断言验证关键不变量。

## 4. Banding LSH 候选生成

将 64 维签名拆为 32 bands × 2 rows；任一 band 完全相等的文档进入同 bucket，形成候选 pair。近似候选概率为 `1-(1-s^r)^b`，呈 S 曲线。增加 rows 提高精度但降召回，增加 bands 提高召回也会增加候选。

bucket key 必须包含 band 编号，防止不同位置的相同四元组误碰撞。

In [ ]:
BANDS71=32; ROWS71=2  # 计算并保存当前步骤的中间状态。
def lsh_candidates71(signatures):  # 定义本节可复用的核心函数。
    buckets=defaultdict(list)  # 计算并保存当前步骤的中间状态。
    for doc_id,sig in sorted(signatures.items()):  # 遍历输入元素以累积或检查结果。
        if sig.shape!=(BANDS71*ROWS71,): raise ValueError("signature_shape")  # 按当前条件选择后续控制路径。
        for band in range(BANDS71): buckets[(band,tuple(int(x) for x in sig[band*ROWS71:(band+1)*ROWS71]))].append(doc_id)  # 遍历输入元素以累积或检查结果。
    pairs=set()  # 计算并保存当前步骤的中间状态。
    for ids in buckets.values():  # 遍历输入元素以累积或检查结果。
        for a,b in combinations(sorted(ids),2): pairs.add((a,b))  # 遍历输入元素以累积或检查结果。
    return pairs,buckets  # 返回当前分支计算出的结果。
candidates71,buckets71=lsh_candidates71(signatures71)  # 计算并保存当前步骤的中间状态。
assert all(a<b for a,b in candidates71) and len(candidates71)<len(docs71)*(len(docs71)-1)//2  # 用受控断言验证关键不变量。
assert ("a","b") in candidates71 and len(buckets71)==BANDS71*len(docs71)-sum(len(v)-1 for v in buckets71.values())  # 用受控断言验证关键不变量。
prob_high71=1-(1-.8**ROWS71)**BANDS71; prob_low71=1-(1-.1**ROWS71)**BANDS71  # 计算并保存当前步骤的中间状态。
assert prob_high71>prob_low71 and .99<prob_high71<=1  # 用受控断言验证关键不变量。

## 5. 精排、阈值与并查集聚类

LSH 只产候选，随后计算 exact Jaccard，达到阈值才连边。连通分量将重复 pair 合成 document family，切分 train/test 时必须按 family 分组，否则同一内容的不同版本会跨 split 泄漏。

传递闭包可能把 A~B、B~C 但 A 不~C 的链合并；生产要根据业务选择 connected component、中心星形或 complete-link。

In [ ]:
class DSU71:  # 定义承载本节状态与行为的数据结构。
    def __init__(self,items): self.parent={x:x for x in items}  # 定义本节可复用的核心函数。
    def find(self,x):  # 定义本节可复用的核心函数。
        while self.parent[x]!=x: self.parent[x]=self.parent[self.parent[x]]; x=self.parent[x]  # 在终止条件满足前持续推进状态。
        return x  # 返回当前分支计算出的结果。
    def union(self,a,b):  # 定义本节可复用的核心函数。
        ra,rb=self.find(a),self.find(b)  # 计算并保存当前步骤的中间状态。
        if ra!=rb: self.parent[max(ra,rb)]=min(ra,rb)  # 按当前条件选择后续控制路径。
threshold71=.38  # 计算并保存当前步骤的中间状态。
duplicate_pairs71={(a,b) for a,b in candidates71 if jaccard71(shingle_map71[a],shingle_map71[b])>=threshold71}  # 计算并保存当前步骤的中间状态。
dsu71=DSU71(shingle_map71)  # 计算并保存当前步骤的中间状态。
for a,b in duplicate_pairs71: dsu71.union(a,b)  # 遍历输入元素以累积或检查结果。
families71=defaultdict(list)  # 计算并保存当前步骤的中间状态。
for item in shingle_map71: families71[dsu71.find(item)].append(item)  # 遍历输入元素以累积或检查结果。
assert ("a","b") in duplicate_pairs71 and ("d","e") in duplicate_pairs71 and ("f","g") in duplicate_pairs71  # 用受控断言验证关键不变量。
assert dsu71.find("a")==dsu71.find("b") and dsu71.find("a")!=dsu71.find("d")  # 用受控断言验证关键不变量。
assert sorted(map(len,families71.values()),reverse=True)[:3]==[2,2,2]  # 用受控断言验证关键不变量。

## 6. Pair-level precision/recall 与候选缩减率

用 exact threshold 在所有 pair 上构造受控 gold，再评估 LSH candidates 的召回和最终 duplicate pair 的 precision/recall。候选缩减率衡量节省了多少精排工作。注意 gold threshold 本身不是人工语义标注，只能验证近似算法没有实现错误。

真实评估应按语言、长度、模板、OCR 噪声和 source 切片，并人工审核阈值附近样本。

In [ ]:
all_pairs71=set(combinations(sorted(shingle_map71),2))  # 计算并保存当前步骤的中间状态。
gold71={(a,b) for a,b in all_pairs71 if jaccard71(shingle_map71[a],shingle_map71[b])>=threshold71}  # 计算并保存当前步骤的中间状态。
def pr71(pred,gold):  # 定义本节可复用的核心函数。
    return (len(pred&gold)/len(pred) if pred else 0.,len(pred&gold)/len(gold) if gold else 0.)  # 返回当前分支计算出的结果。
candidate_precision71,candidate_recall71=pr71(candidates71,gold71); final_precision71,final_recall71=pr71(duplicate_pairs71,gold71)  # 计算并保存当前步骤的中间状态。
reduction71=1-len(candidates71)/len(all_pairs71)  # 计算并保存当前步骤的中间状态。
assert candidate_recall71==1. and final_precision71==1. and final_recall71==1.  # 用受控断言验证关键不变量。
assert 0<reduction71<1 and candidate_precision71<=final_precision71  # 用受控断言验证关键不变量。
assert gold71==duplicate_pairs71  # 用受控断言验证关键不变量。

## 7. 增量、版本与删除

新文档只需算 signature 并探测对应 buckets；但 family 可能改变。更新同一 doc ID 要写新版本并 tombstone 旧签名；从 bucket 删除需要倒排记录，或查询时按 latest 过滤并后台 compact。

MinHash 参数不变时可增量；一旦改变 shingle/seed/bands，必须全量重建，不能让一个 bucket 混合两种签名语义。

In [ ]:
new_doc71=Doc71("i",1,"s7","vector search uses an inverted file with product quantization")  # 计算并保存当前步骤的中间状态。
new_shingles71=shingles71(new_doc71.text); new_signature71=signature71(new_shingles71)  # 计算并保存当前步骤的中间状态。
extended_signatures71={**signatures71,"i":new_signature71}; extended_candidates71,_=lsh_candidates71(extended_signatures71)  # 计算并保存当前步骤的中间状态。
assert any("i" in pair for pair in extended_candidates71) and new_signature71.shape==(64,)  # 用受控断言验证关键不变量。
latest71={d.doc_id:d.version for d in docs71}; deleted71=set(); latest71["i"]=1; latest71["a"]=2; deleted71.add("a")  # 计算并保存当前步骤的中间状态。
live_candidates71={p for p in extended_candidates71 if all(x not in deleted71 for x in p)}  # 计算并保存当前步骤的中间状态。
assert all("a" not in p for p in live_candidates71) and any("i" in p for p in live_candidates71)  # 用受控断言验证关键不变量。

## 8. 快照、信任边界与面试总结

manifest 绑定 normalization、shingle n、stable hash、prime、全部 `a/b`、bands/rows、阈值、文档版本和 signature bytes。loader 从实际数组重算摘要并与外部 registry 比较。

面试回答应强调两阶段：LSH 保召回地产候选，exact/模型精排保精度；然后讨论 family split、增量与阈值审核，而不是宣称 MinHash 能直接判重复。

In [ ]:
def sig_digest71(sigs):  # 定义本节可复用的核心函数。
    h=hashlib.sha256()  # 计算并保存当前步骤的中间状态。
    for key,val in sorted(sigs.items()): h.update(key.encode()); h.update(str(val.dtype).encode()); h.update(np.ascontiguousarray(val).tobytes())  # 遍历输入元素以累积或检查结果。
    return h.hexdigest()  # 返回当前分支计算出的结果。
manifest71={"artifact_id":"minhash-lsh-v1","normalization":"NFKC-lower-word-v1","shingle_n":3,"num_perm":NUM_PERM71,"prime":PRIME71,"a_sha":sha71(A71.tobytes()),"b_sha":sha71(B71.tobytes()),"bands":BANDS71,"rows":ROWS71,"threshold":threshold71,"signature_digest":sig_digest71(signatures71)}  # 计算并保存当前步骤的中间状态。
TRUST71=MappingProxyType({manifest71["artifact_id"]:sha71(canonical71(manifest71).encode())})  # 计算并保存当前步骤的中间状态。
def load_lsh71(m,sigs):  # 定义本节可复用的核心函数。
    actual=copy.deepcopy(m); actual["signature_digest"]=sig_digest71(sigs)  # 计算并保存当前步骤的中间状态。
    if TRUST71.get(actual.get("artifact_id"))!=sha71(canonical71(actual).encode()): raise RuntimeError("untrusted_lsh_snapshot")  # 按当前条件选择后续控制路径。
    return MappingProxyType(actual)  # 返回当前分支计算出的结果。
pub71=load_lsh71(manifest71,signatures71)  # 计算并保存当前步骤的中间状态。
assert pub71["num_perm"]==64 and isinstance(TRUST71,MappingProxyType)  # 用受控断言验证关键不变量。
forged71={k:v.copy() for k,v in signatures71.items()}; forged71["a"][0]+=1  # 计算并保存当前步骤的中间状态。
try: load_lsh71(manifest71,forged71); raise AssertionError("forged signature accepted")  # 尝试执行可能失败的受控操作。
except RuntimeError as e: assert str(e)=="untrusted_lsh_snapshot"  # 捕获预期异常并验证失败分支。
print({"all_pairs":len(all_pairs71),"candidates":len(candidates71),"candidate_recall":candidate_recall71,"reduction":round(reduction71,3)})  # 执行当前语句以推进本节示例。

## 9. 复杂度、失败模式与来源

签名成本约 `O(N * shingles * permutations)`，band 插入约 `O(N*b)`，精排成本取决于候选量。常见错误：使用 Python hash、空文档签名、把 LSH 碰撞当最终重复、跨版本参数混用、family 跨 split、链式过合并和删除后旧 bucket 复活。

- Broder, [On the resemblance and containment of documents](https://www.cs.princeton.edu/courses/archive/spring13/cos598C/broder97resemblance.pdf), 1997。
- Leskovec et al., [Mining of Massive Datasets: Locality-Sensitive Hashing](http://www.mmds.org/mmds/v2.1/ch03-lsh.pdf)。
- Shrivastava & Li, [Densifying One Permutation Hashing](https://proceedings.mlr.press/v32/shrivastava14.html)，扩展阅读。